# SpaceX Falcon 9 — Geospatial Analysis Using Folium

## Objective

Visualize SpaceX launch sites on interactive maps and analyze
their geographical characteristics and landing classifications.

Dataset: Historical Falcon 9 launches (2010–2020)

In [1]:
# Step 1: Import geospatial analysis libraries

import pandas as pd
import folium                    #Creates interactive geographical maps

from folium.plugins import MarkerCluster

print("Libraries imported successfully!")
print("Pandas version:", pd.__version__)
print("Folium version:", folium.__version__)

Libraries imported successfully!
Pandas version: 2.2.3
Folium version: 0.20.0


## Step 2 — Load and Inspect Launch Site Data

Load the historical SpaceX dataset and inspect the unique
launch sites and their geographical coordinates.


In [2]:
# Step 2: Load and inspect SpaceX launch site data

from google.colab import files

uploaded = files.upload()

df = pd.read_csv("spacex_launches_processed.csv")

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

print("\nLaunch sites and geographical coordinates:")

launch_sites = (
    df.groupby("LaunchSite")
    .agg(
        Latitude=("Latitude", "first"),
        Longitude=("Longitude", "first"),
        TotalLaunches=("FlightNumber", "count")
    )
    .reset_index()
)

display(launch_sites)

print("\nMissing geographical coordinates:")
print(df[["Latitude", "Longitude"]].isnull().sum())

Saving spacex_launches_processed.csv to spacex_launches_processed.csv
Dataset loaded successfully!
Dataset shape: (90, 18)

Launch sites and geographical coordinates:


,LaunchSite,Latitude,Longitude,TotalLaunches
0,CCAFS SLC 40,28.561857,-80.577366,55
1,KSC LC 39A,28.608058,-80.603956,22
2,VAFB SLC 4E,34.632093,-120.610829,13



Missing geographical coordinates:
Latitude     0
Longitude    0
dtype: int64


## Step 3 — Interactive Map of SpaceX Launch Sites

Create an interactive Folium map displaying the three historical
SpaceX launch sites with geographical markers and launch counts.

In [5]:
# Step 3: Create a SpaceX launch site map without external tiles

spacex_map = folium.Map(
    location=[37.0, -96.0],
    zoom_start=4,
    tiles=None
)

# Add a marker for each launch site
for _, site in launch_sites.iterrows():

    folium.Marker(
        location=[site["Latitude"], site["Longitude"]],

        popup=(
            f"Launch Site: {site['LaunchSite']}<br>"
            f"Total Launches: {site['TotalLaunches']}"
        ),

        tooltip=site["LaunchSite"],

        icon=folium.Icon(
            color="blue",
            icon="info-sign"
        )
    ).add_to(spacex_map)

# Display the map
spacex_map

In [7]:
# Step 3A: Add an alternative geographical background

folium.TileLayer(
    tiles=(
        "https://server.arcgisonline.com/ArcGIS/rest/services/"
        "World_Imagery/MapServer/tile/{z}/{y}/{x}"
    ),
    attr="Tiles © Esri and contributors",
    name="Esri World Imagery",
    overlay=False,
    control=True
).add_to(spacex_map)

folium.LayerControl().add_to(spacex_map)

spacex_map

## Step 4 — Launch Site Circle Markers

Add interactive circle markers to identify SpaceX launch sites
and display their historical launch counts.

In [8]:
# Step 4: Add circle markers for each launch site

for _, site in launch_sites.iterrows():

    folium.CircleMarker(
        location=[site["Latitude"], site["Longitude"]],
        radius=12,
        color="yellow",
        fill=True,
        fill_color="yellow",
        fill_opacity=0.6,

        popup=folium.Popup(
            f"Launch Site: {site['LaunchSite']}<br>"
            f"Total Launches: {site['TotalLaunches']}",
            max_width=250
        ),

        tooltip=site["LaunchSite"]

    ).add_to(spacex_map)

# Display updated map
spacex_map

## Step 5 — Landing Classification Marker Clusters

Visualize historical Falcon 9 launch records using
color-coded markers and MarkerCluster.

Green = Class 1
Red = Class 0

In [9]:
# Step 5: Create landing classification marker clusters

landing_map = folium.Map(
    location=[37.0, -96.0],
    zoom_start=4,
    tiles=None
)

# Add satellite imagery
folium.TileLayer(
    tiles=(
        "https://server.arcgisonline.com/ArcGIS/rest/services/"
        "World_Imagery/MapServer/tile/{z}/{y}/{x}"
    ),
    attr="Tiles © Esri and contributors",
    name="Esri World Imagery"
).add_to(landing_map)

# Create a marker cluster
marker_cluster = MarkerCluster().add_to(landing_map)

# Add each launch record
for _, launch in df.iterrows():

    marker_color = (
        "green" if launch["Class"] == 1 else "red"
    )

    folium.Marker(
        location=[
            launch["Latitude"],
            launch["Longitude"]
        ],

        popup=(
            f"Flight Number: {launch['FlightNumber']}<br>"
            f"Launch Site: {launch['LaunchSite']}<br>"
            f"Orbit: {launch['Orbit']}<br>"
            f"Class: {launch['Class']}"
        ),

        icon=folium.Icon(
            color=marker_color,
            icon="info-sign"
        )

    ).add_to(marker_cluster)

# Display the map
landing_map

## Step 6 — Geographical Landing Classification Distribution

Summarize Class 0 and Class 1 records by launch site
and visualize the results using interactive circle markers.

In [10]:
# Step 6: Summarize landing classifications by launch site

site_class_summary = (
    df.groupby("LaunchSite")
    .agg(
        Latitude=("Latitude", "first"),
        Longitude=("Longitude", "first"),
        TotalLaunches=("Class", "count"),
        Class1Count=("Class", "sum")
    )
    .reset_index()
)

site_class_summary["Class0Count"] = (
    site_class_summary["TotalLaunches"]
    - site_class_summary["Class1Count"]
)

site_class_summary["Class1Percentage"] = (
    site_class_summary["Class1Count"]
    / site_class_summary["TotalLaunches"] * 100
).round(2)

print("Geographical landing classification summary:")
display(site_class_summary)

Geographical landing classification summary:


,LaunchSite,Latitude,Longitude,TotalLaunches,Class1Count,Class0Count,Class1Percentage
0,CCAFS SLC 40,28.561857,-80.577366,55,47,8,85.45
1,KSC LC 39A,28.608058,-80.603956,22,21,1,95.45
2,VAFB SLC 4E,34.632093,-120.610829,13,11,2,84.62


## Step 7 — Interactive Launch Site Classification Map

Create a geographical visualization of Class 0 and Class 1
distributions across the three SpaceX launch sites.

Display launch counts and Class 1 percentages in interactive popups.

In [11]:
# Step 7: Create an interactive classification summary map

classification_map = folium.Map(
    location=[37.0, -96.0],
    zoom_start=4,
    tiles=None
)

# Add satellite imagery
folium.TileLayer(
    tiles=(
        "https://server.arcgisonline.com/ArcGIS/rest/services/"
        "World_Imagery/MapServer/tile/{z}/{y}/{x}"
    ),
    attr="Tiles © Esri and contributors",
    name="Esri World Imagery"
).add_to(classification_map)

# Add a circle marker for each launch site
for _, site in site_class_summary.iterrows():

    popup_text = (
        f"<b>Launch Site:</b> {site['LaunchSite']}<br>"
        f"<b>Total Launches:</b> {site['TotalLaunches']}<br>"
        f"<b>Class 1:</b> {site['Class1Count']}<br>"
        f"<b>Class 0:</b> {site['Class0Count']}<br>"
        f"<b>Class 1 Percentage:</b> {site['Class1Percentage']:.2f}%"
    )

    folium.CircleMarker(
        location=[site["Latitude"], site["Longitude"]],
        radius=14,
        color="blue",
        fill=True,
        fill_color="green",
        fill_opacity=0.7,
        popup=folium.Popup(popup_text, max_width=300),
        tooltip=site["LaunchSite"]
    ).add_to(classification_map)

# Display interactive map
classification_map

## Step 8 — Geographical Distance Analysis

Calculate the approximate great-circle distance between
CCAFS SLC 40 and KSC LC 39A using the Haversine formula.


In [12]:
# Step 8: Calculate geographical distance

from math import radians, sin, cos, sqrt, atan2

def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371  # Earth's approximate radius in kilometers

    lat1, lon1 = radians(lat1), radians(lon1)
    lat2, lon2 = radians(lat2), radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        sin(dlat / 2)**2
        + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    )

    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    return R * c


# Extract coordinates from our launch-site summary
ccafs = site_class_summary[
    site_class_summary["LaunchSite"] == "CCAFS SLC 40"
].iloc[0]

ksc = site_class_summary[
    site_class_summary["LaunchSite"] == "KSC LC 39A"
].iloc[0]

distance_km = haversine_distance(
    ccafs["Latitude"],
    ccafs["Longitude"],
    ksc["Latitude"],
    ksc["Longitude"]
)

print(f"Distance between CCAFS SLC 40 and KSC LC 39A: {distance_km:.2f} km")

Distance between CCAFS SLC 40 and KSC LC 39A: 5.76 km


## Step 9 — Visualizing Geographical Distance

Create an interactive map connecting CCAFS SLC 40 and
KSC LC 39A using Folium PolyLine.

Display their approximate great-circle distance.

In [13]:
# Step 9: Visualize distance between Florida launch sites

florida_map = folium.Map(
    location=[28.585, -80.59],
    zoom_start=12,
    tiles=None
)

# Add satellite imagery
folium.TileLayer(
    tiles=(
        "https://server.arcgisonline.com/ArcGIS/rest/services/"
        "World_Imagery/MapServer/tile/{z}/{y}/{x}"
    ),
    attr="Tiles © Esri and contributors",
    name="Esri World Imagery"
).add_to(florida_map)

# Store both launch-site coordinates
ccafs_coords = [ccafs["Latitude"], ccafs["Longitude"]]
ksc_coords = [ksc["Latitude"], ksc["Longitude"]]

# Add markers for both launch sites
folium.Marker(
    location=ccafs_coords,
    popup="CCAFS SLC 40",
    tooltip="CCAFS SLC 40",
    icon=folium.Icon(color="blue")
).add_to(florida_map)

folium.Marker(
    location=ksc_coords,
    popup="KSC LC 39A",
    tooltip="KSC LC 39A",
    icon=folium.Icon(color="green")
).add_to(florida_map)

# Connect the two sites
folium.PolyLine(
    locations=[ccafs_coords, ksc_coords],
    color="red",
    weight=4,
    opacity=0.8,
    tooltip=f"Distance: {distance_km:.2f} km"
).add_to(florida_map)

# Display the map
florida_map

## Step 10 — Final Geospatial Analysis Validation

Validate the geographical dataset, launch-site statistics,
landing classification counts, and Haversine distance.


In [14]:
# Step 10: Final geospatial validation

print("Total launch records:", len(df))
print("Unique launch sites:", df["LaunchSite"].nunique())

print("\nLanding classification totals:")
print(df["Class"].value_counts().sort_index())

print("\nMissing geographical coordinates:")
print(df[["Latitude", "Longitude"]].isnull().sum())

print("\nLaunch site summary:")
display(site_class_summary)

print(f"\nFlorida launch site distance: {distance_km:.2f} km")

# Validate the results
assert len(df) == 90
assert df["LaunchSite"].nunique() == 3
assert df["Class"].value_counts().to_dict() == {0: 11, 1: 79}
assert df[["Latitude", "Longitude"]].notnull().all().all()
assert site_class_summary["TotalLaunches"].sum() == 90
assert site_class_summary["Class1Count"].sum() == 79
assert site_class_summary["Class0Count"].sum() == 11
assert 5.7 < distance_km < 5.9

print("\nAll geospatial validation checks passed!")

Total launch records: 90
Unique launch sites: 3

Landing classification totals:
Class
0    11
1    79
Name: count, dtype: int64

Missing geographical coordinates:
Latitude     0
Longitude    0
dtype: int64

Launch site summary:


,LaunchSite,Latitude,Longitude,TotalLaunches,Class1Count,Class0Count,Class1Percentage
0,CCAFS SLC 40,28.561857,-80.577366,55,47,8,85.45
1,KSC LC 39A,28.608058,-80.603956,22,21,1,95.45
2,VAFB SLC 4E,34.632093,-120.610829,13,11,2,84.62



Florida launch site distance: 5.76 km

All geospatial validation checks passed!


## Step 11 — Export Interactive Folium Maps

Save the generated geographical visualizations as HTML files
for the SpaceX capstone project and GitHub repository.

In [15]:
# Step 11: Save interactive Folium maps

import os

os.makedirs("maps", exist_ok=True)

spacex_map.save("maps/spacex_launch_sites.html")

landing_map.save("maps/spacex_landing_clusters.html")

classification_map.save("maps/spacex_classification_map.html")

florida_map.save("maps/spacex_florida_distance.html")

print("All four interactive maps saved successfully!")

for filename in os.listdir("maps"):
    filepath = os.path.join("maps", filename)
    print(filename, "-", os.path.getsize(filepath), "bytes")

All four interactive maps saved successfully!
spacex_classification_map.html - 7532 bytes
spacex_landing_clusters.html - 122984 bytes
spacex_florida_distance.html - 7082 bytes
spacex_launch_sites.html - 13650 bytes


In [16]:
# Step 12: Package and download all four interactive maps

import shutil
from google.colab import files

shutil.make_archive(
    "SpaceX_Folium_Maps",
    "zip",
    "maps"
)

files.download("SpaceX_Folium_Maps.zip")

print("Map archive created successfully!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Map archive created successfully!
